In [ ]:
# TV OFF / ACE-Step 1.5 — non-publishing GPU smoke test
# This notebook only creates a test audio file and validation manifest.
# It does not touch the TV OFF publishing queue or YouTube credentials.

import os
import sys
import json
import math
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "ACE-Step-1.5"
OUT = WORK / "ace_step_output"
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env, check=True)

# Hardware preflight. Kaggle must provide one CUDA-visible Tesla T4.
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing a CPU fallback for this GPU test.")
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print("GPU", index, torch.cuda.get_device_name(index),
          round(torch.cuda.get_device_properties(index).total_memory / 1024**3, 2), "GiB")

if not any("T4" in torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())):
    raise RuntimeError("Expected a Tesla T4 accelerator; detected a different GPU.")

# Fetch the official upstream project at a pinned release tag, not an unpinned branch.
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", "v1.5.2",
         "https://github.com/ACE-Step/ACE-Step-1.5.git", REPO])

# Install the official project dependencies into the Kaggle Python environment.
# uv follows the project's declared CUDA 12.8 PyTorch wheel source on Linux.
uv = Path.home() / ".local" / "bin" / "uv"
if not uv.exists():
    run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])
if not uv.exists():
    raise RuntimeError("uv installer completed but the executable was not found.")
run([uv, "pip", "install", "--system", "-e", "."], cwd=REPO)

# Import after dependency installation.
from acestep.handler import AceStepHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music
import soundfile as sf
import numpy as np

# One T4 has 16 GiB physical memory (about 14.5 GiB usable in the previous
# successful TV OFF smoke test). Keep batch=1, use the 2B Turbo DiT, CPU offload,
# and INT8 weight-only quantization. Do not attempt the XL model.
dit = AceStepHandler()
message, initialized = dit.initialize_service(
    project_root=str(REPO),
    config_path="acestep-v15-turbo",
    device="cuda",
    offload_to_cpu=True,
    quantization="int8_weight_only",
)
print("ACE-Step initialization:", message)
if not initialized:
    raise RuntimeError("ACE-Step DiT initialization failed.")

# DiT-only inference avoids loading a second large LM on the first T4 test.
# We explicitly request instrumental music and disable all LM/CoT paths.
params = GenerationParams(
    task_type="text2music",
    caption=(
        "Original instrumental downtempo ambient electronic composition, "
        "warm felt piano playing a memorable evolving melody, gentle analog "
        "synth arpeggios, deep rounded bass, layered harmonic progression, "
        "subtle rhythmic pulse, changing musical sections, expressive dynamics, "
        "professional studio production, no vocals, no speech, no sound effects"
    ),
    lyrics="[Instrumental]",
    instrumental=True,
    vocal_language="unknown",
    bpm=82,
    duration=30,
    inference_steps=8,
    shift=3.0,
    seed=20261120,
    thinking=False,
    use_cot_metas=False,
    use_cot_caption=False,
    use_cot_language=False,
)
config = GenerationConfig(
    batch_size=1,
    allow_lm_batch=False,
    use_random_seed=False,
    seeds=[20261120],
    audio_format="wav",
)
result = generate_music(dit, None, params, config, save_dir=str(OUT))
if not result.success:
    raise RuntimeError(f"ACE-Step generation failed: {result.error}")

if not result.audios:
    raise RuntimeError("ACE-Step returned success but no audio outputs.")

audio_source = Path(result.audios[0]["path"])
if not audio_source.is_file():
    raise RuntimeError(f"ACE-Step audio path does not exist: {audio_source}")
audio_target = OUT / "tv_off_ace_step_t4_test.wav"
if audio_source.resolve() != audio_target.resolve():
    shutil.copy2(audio_source, audio_target)

info = sf.info(str(audio_target))
audio, sr = sf.read(str(audio_target), always_2d=True, dtype="float32")
peak = float(np.max(np.abs(audio)))
rms = float(np.sqrt(np.mean(np.square(audio))))
duration = float(info.duration)
if duration < 25 or duration > 35:
    raise RuntimeError(f"Unexpected audio duration: {duration:.2f}s")
if not np.isfinite(audio).all():
    raise RuntimeError("Audio contains NaN or infinite samples.")
if peak < 0.01:
    raise RuntimeError(f"Audio is nearly silent (peak={peak:.6f}).")
if peak > 1.001:
    raise RuntimeError(f"Audio clips above full scale (peak={peak:.4f}).")
if rms < 0.001:
    raise RuntimeError(f"Audio RMS is too low (rms={rms:.6f}).")

manifest = {
    "status": "ok",
    "provider": "ACE-Step-1.5",
    "model": "acestep-v15-turbo",
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "cuda": True,
    "duration_seconds": round(duration, 3),
    "sample_rate": int(sr),
    "channels": int(info.channels),
    "subtype": info.subtype,
    "peak": round(peak, 6),
    "rms": round(rms, 6),
    "seed": 20261120,
    "instrumental": True,
    "publishing_enabled": False,
    "audio_file": audio_target.name,
}
(OUT / "ace_step_smoke_result.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, indent=2))
print("TEST PASSED — audio generated and validated; nothing was published.")
